<a href="https://colab.research.google.com/github/mas622424/WISER-BQP-QAPINN/blob/main/notebooks/01_cPINN_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import numpy as np

# 1. Same Physics Residual Function
def compute_burgers_residual(model, x, t, nu=0.01/np.pi):
    x.requires_grad_(True)
    t.requires_grad_(True)
    u = model(torch.cat([x, t], dim=1))
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), retain_graph=True, create_graph=True)[0]
    return u_t + u * u_x - nu * u_xx

# 2. Classical PINN Architecture (No Quantum Layer)
class ClassicalPINN(nn.Module):
    def __init__(self):
        super(ClassicalPINN, self).__init__()
        # We replace the 4-qubit quantum layer with a classical layer of the same size
        self.input_layer = nn.Linear(2, 4)
        self.classical_hidden = nn.Linear(4, 4) # The replacement layer
        self.hidden1 = nn.Linear(4, 20)
        self.act1 = nn.Tanh()
        self.hidden2 = nn.Linear(20, 20)
        self.act2 = nn.Tanh()
        self.output_layer = nn.Linear(20, 1)

    def forward(self, x_t):
        out = torch.tanh(self.input_layer(x_t))
        out = torch.tanh(self.classical_hidden(out)) # Classical operation instead of quantum
        out = self.act1(self.hidden1(out))
        out = self.act2(self.hidden2(out))
        return self.output_layer(out)

# 3. Initialize Model and Print Parameter Count
cpinn_model = ClassicalPINN()
total_params = sum(p.numel() for p in cpinn_model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters in Classical PINN: {total_params}")

Total Trainable Parameters in Classical PINN: 573
